# Train Multi-Class Prostate Organ Segmentation (5-Fold Cross-Validation)

This notebook runs 5-fold cross-validation training for **multi-class organ segmentation** using RRUNet3D.

**Segmentation Classes:**
- 0 = Background
- 1 = TZ (Transition Zone)
- 2 = PZ (Peripheral Zone)

**Prerequisites:**
- Ensure `data/preprocessed/t2/` and `data/preprocessed/organ_masks/` contain `<SUBJECT_ID>.nii.gz` pairs.
- Organ masks should be multi-class: 0=background, 1=TZ, 2=PZ
- Fold-specific CSV files will be created automatically from `data/train.csv` if needed.
- Adjust `configs/organ.yaml` if needed (should have `out_channels: 3` for multi-class).


In [1]:
!pip list 

Package                            Version
---------------------------------- ------------
absl-py                            2.2.2
alembic                            1.15.2
annotated-types                    0.7.0
anyio                              4.9.0
astor                              0.8.1
asttokens                          3.0.0
attrs                              25.3.0
beautifulsoup4                     4.13.4
blinker                            1.9.0
cachetools                         5.5.2
certifi                            2025.4.26
charset-normalizer                 3.4.2
clearml                            2.0.0rc0
click                              8.1.8
cloudpickle                        2.2.1
colorama                           0.4.6
colorlog                           6.9.0
comm                               0.2.2
contourpy                          1.3.2
crc32c                             2.7.1
cycler                             0.12.1
databricks-sdk                     0.

In [2]:
%load_ext autoreload
%autoreload 2
import os, sys
repo_root = os.path.abspath("..") if os.getcwd().endswith("notebooks") else os.path.abspath(".")
if os.getcwd().endswith("notebooks"):
    os.chdir(repo_root)
print("CWD:", os.getcwd())


CWD: /home/anson/work/research-contributions/prostate-mri-lesion-seg


In [3]:
# Setup: Create fold-specific CSV files if needed
from pathlib import Path
import pandas as pd

splits_dir = Path("annotations/splits")
splits_dir.mkdir(parents=True, exist_ok=True)

# Check if CSV files exist
missing_folds = []
for fold in range(5):
    if not (splits_dir / f"fold{fold}_train.csv").exists():
        missing_folds.append(fold)

if missing_folds:
    print(f"Creating fold-specific CSV files for folds: {missing_folds}")
    df = pd.read_csv("data/train.csv")
    train_df = df[df['fold'] >= 0].copy()
    
    print(f"Total training patients: {len(train_df)}")
    print(f"Fold distribution: {train_df['fold'].value_counts().sort_index().to_dict()}")
    
    for fold in range(5):
        val_mask = train_df['fold'] == fold
        val_ids = train_df[val_mask]['ID'].tolist()
        train_mask = train_df['fold'] != fold
        train_ids = train_df[train_mask]['ID'].tolist()
        
        print(f"Fold {fold}: Train={len(train_ids)}, Val={len(val_ids)}")
        
        train_csv_df = pd.DataFrame({'subject_id': train_ids})
        train_csv_df.to_csv(splits_dir / f"fold{fold}_train.csv", index=False)
        
        val_csv_df = pd.DataFrame({'subject_id': val_ids})
        val_csv_df.to_csv(splits_dir / f"fold{fold}_val.csv", index=False)
    
    print(f"\n✓ Created fold-specific CSV files in {splits_dir}")
else:
    print("✓ Fold-specific CSV files already exist")


✓ Fold-specific CSV files already exist


In [3]:
# 5-Fold Cross-Validation Training
from training.engine import train_organ_from_config
from training.utils import load_yaml
from pathlib import Path
import logging
import copy
import yaml
import tempfile
import os

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

config_path = "configs/organ.yaml"
splits_dir = Path("annotations/splits")

# Load base config
cfg = load_yaml(config_path)
print(f"Model: {cfg['model']['type']} with {cfg['model']['out_channels']} output channels (multi-class)")
print(f"Training all 5 folds...\n")

# Train all folds
all_artifacts = []
for fold in range(5):
    print(f"\n{'='*60}")
    print(f"Starting Fold {fold} Training")
    print(f"{'='*60}")
    
    # Create a deep copy of config for this fold
    fold_cfg = copy.deepcopy(cfg)
    
    # Update config to use fold-specific CSV files
    train_csv = splits_dir / f"fold{fold}_train.csv"
    val_csv = splits_dir / f"fold{fold}_val.csv"
    
    fold_cfg["dataset"]["train_split_csv"] = str(train_csv)
    fold_cfg["dataset"]["val_split_csv"] = str(val_csv)
    fold_cfg["dataset"]["scan_all"] = False  # Use CSV files
    
    # Update output directory to include fold number
    base_exp_dir = fold_cfg["output"]["exp_dir"]
    fold_cfg["output"]["exp_dir"] = str(Path(base_exp_dir) / f"fold{fold}")
    
    print(f"Train CSV: {train_csv}")
    print(f"Val CSV: {val_csv}")
    print(f"Output directory: {fold_cfg['output']['exp_dir']}\n")
    
    # Create temporary YAML file for this fold (since train_organ_from_config expects a file path)
    with tempfile.NamedTemporaryFile(mode='w', suffix='.yaml', delete=False) as tmp_file:
        yaml.dump(fold_cfg, tmp_file, default_flow_style=False)
        tmp_config_path = tmp_file.name
    
    try:
        # Train using the temporary config file
        artifacts = train_organ_from_config(tmp_config_path)
        all_artifacts.append((fold, artifacts))
        print(f"\n✓ Fold {fold} training completed!")
        print(f"  Best checkpoint: {artifacts.best_ckpt}")
        print(f"  Metrics CSV: {artifacts.metrics_csv}")
    except Exception as e:
        print(f"\n✗ Fold {fold} training failed: {e}")
        import traceback
        traceback.print_exc()
        continue
    finally:
        # Clean up temporary config file
        if os.path.exists(tmp_config_path):
            os.unlink(tmp_config_path)

# Summary
print(f"\n{'='*60}")
print("5-Fold Cross-Validation Training Summary")
print(f"{'='*60}")
print(f"Successfully trained: {len(all_artifacts)}/5 folds")
for fold, artifacts in all_artifacts:
    print(f"  Fold {fold}: {artifacts.best_ckpt}")

# Store artifacts for later use
fold_artifacts = {fold: artifacts for fold, artifacts in all_artifacts}

Model: rrunet3d with 3 output channels (multi-class)
Training all 5 folds...


Starting Fold 0 Training
Train CSV: annotations/splits/fold0_train.csv
Val CSV: annotations/splits/fold0_val.csv
Output directory: experiments/organ/fold0

Using device: cuda


Loading dataset: 100%|██████████| 26/26 [00:03<00:00,  8.29it/s]


Epoch 1/200


`torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
`torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.


  step 10/65 - loss: 1.7434
  step 20/65 - loss: 1.6830
  step 30/65 - loss: 1.6593
  step 40/65 - loss: 1.6392
  step 50/65 - loss: 1.6192
  step 60/65 - loss: 1.6022
  val mean dice: 0.2055
Epoch 2/200
  step 10/65 - loss: 1.4930
  step 20/65 - loss: 1.4869
  step 30/65 - loss: 1.4839
  step 40/65 - loss: 1.4745
  step 50/65 - loss: 1.4597
  step 60/65 - loss: 1.4467
  val mean dice: 0.2371
Epoch 3/200
  step 10/65 - loss: 1.3822
  step 20/65 - loss: 1.3802
  step 30/65 - loss: 1.3771
  step 40/65 - loss: 1.3735
  step 50/65 - loss: 1.3666
  step 60/65 - loss: 1.3587
  val mean dice: 0.3159
Epoch 4/200
  step 10/65 - loss: 1.3363
  step 20/65 - loss: 1.3287
  step 30/65 - loss: 1.3298
  step 40/65 - loss: 1.3289
  step 50/65 - loss: 1.3262
  step 60/65 - loss: 1.3214
  val mean dice: 0.3935
Epoch 5/200
  step 10/65 - loss: 1.3052
  step 20/65 - loss: 1.3047
  step 30/65 - loss: 1.3058
  step 40/65 - loss: 1.3068
  step 50/65 - loss: 1.3054
  step 60/65 - loss: 1.3018
  val mean dice:

...

Epoch 1/200


  step 10/65 - loss: 1.7482
  step 20/65 - loss: 1.6957
  step 30/65 - loss: 1.6700
  step 40/65 - loss: 1.6457
  step 50/65 - loss: 1.6257
  step 60/65 - loss: 1.6077
  val mean dice: 0.1927
Epoch 2/200
  step 10/65 - loss: 1.4831
  step 20/65 - loss: 1.4835
  step 30/65 - loss: 1.4782
  step 40/65 - loss: 1.4701
  step 50/65 - loss: 1.4571
  step 60/65 - loss: 1.4442
  val mean dice: 0.2666
Epoch 3/200
  step 10/65 - loss: 1.3664
  step 20/65 - loss: 1.3728
  step 30/65 - loss: 1.3675
  step 40/65 - loss: 1.3652
  step 50/65 - loss: 1.3595
  step 60/65 - loss: 1.3551
  val mean dice: 0.3309
Epoch 4/200
  step 10/65 - loss: 1.3317
  step 20/65 - loss: 1.3285
  step 30/65 - loss: 1.3291
  step 40/65 - loss: 1.3264
  step 50/65 - loss: 1.3218
  step 60/65 - loss: 1.3170
  val mean dice: 0.3647
Epoch 5/200
  step 10/65 - loss: 1.3108
  step 20/65 - loss: 1.3028
  step 30/65 - loss: 1.3051
  step 40/65 - loss: 1.3015
  step 50/65 - loss: 1.3028
  step 60/65 - loss: 1.3001
  val mean dice:

Loading dataset: 100%|██████████| 26/26 [00:03<00:00,  6.90it/s]


Epoch 1/200
  step 10/65 - loss: 1.7504
  step 20/65 - loss: 1.6951
  step 30/65 - loss: 1.6692
  step 40/65 - loss: 1.6472
  step 50/65 - loss: 1.6279
  step 60/65 - loss: 1.6093
  val mean dice: 0.2025
Epoch 2/200
  step 10/65 - loss: 1.4889
  step 20/65 - loss: 1.4861
  step 30/65 - loss: 1.4789
  step 40/65 - loss: 1.4700
  step 50/65 - loss: 1.4575
  step 60/65 - loss: 1.4466
  val mean dice: 0.2259
Epoch 3/200
  step 10/65 - loss: 1.3801
  step 20/65 - loss: 1.3802
  step 30/65 - loss: 1.3777
  step 40/65 - loss: 1.3746
  step 50/65 - loss: 1.3680
  step 60/65 - loss: 1.3627
  val mean dice: 0.3285
Epoch 4/200
  step 10/65 - loss: 1.3341
  step 20/65 - loss: 1.3365
  step 30/65 - loss: 1.3350
  step 40/65 - loss: 1.3287
  step 50/65 - loss: 1.3244
  step 60/65 - loss: 1.3208
  val mean dice: 0.4236
Epoch 5/200
  step 10/65 - loss: 1.3067
  step 20/65 - loss: 1.3119
  step 30/65 - loss: 1.3146
  step 40/65 - loss: 1.3101
  step 50/65 - loss: 1.3073
  step 60/65 - loss: 1.3027
  va

KeyboardInterrupt: 

## Fine-tune from Pretrained Weights (5-Fold Cross-Validation)

Loads the existing TorchScript model from `models/organ/model.ts` and continues training on all 5 folds.
The pretrained weights are used as initialization instead of random weights; everything else
(loss, optimizer, scheduler, checkpointing, experiment directories) remains identical.

In [4]:
# 5-Fold Cross-Validation Training (from pretrained weights)
from training.engine import train_organ_from_config
from training.utils import load_yaml
from pathlib import Path
import copy
import yaml
import tempfile
import os

pretrained_path = "models/organ/model.ts"
config_path = "configs/organ.yaml"
splits_dir = Path("annotations/splits")

cfg = load_yaml(config_path)
print(f"Model: {cfg['model']['type']} with {cfg['model']['out_channels']} output channels (multi-class)")
print(f"Pretrained weights: {pretrained_path}")
print(f"Training all 5 folds...\n")

all_artifacts = []
for fold in range(5):
    print(f"\n{'='*60}")
    print(f"Starting Fold {fold} Training (pretrained)")
    print(f"{'='*60}")

    fold_cfg = copy.deepcopy(cfg)

    train_csv = splits_dir / f"fold{fold}_train.csv"
    val_csv = splits_dir / f"fold{fold}_val.csv"

    fold_cfg["dataset"]["train_split_csv"] = str(train_csv)
    fold_cfg["dataset"]["val_split_csv"] = str(val_csv)
    fold_cfg["dataset"]["scan_all"] = False

    base_exp_dir = fold_cfg["output"]["exp_dir"]
    fold_cfg["output"]["exp_dir"] = str(Path(base_exp_dir) / f"fold{fold}")

    print(f"Train CSV: {train_csv}")
    print(f"Val CSV: {val_csv}")
    print(f"Output directory: {fold_cfg['output']['exp_dir']}\n")

    with tempfile.NamedTemporaryFile(mode='w', suffix='.yaml', delete=False) as tmp_file:
        yaml.dump(fold_cfg, tmp_file, default_flow_style=False)
        tmp_config_path = tmp_file.name

    try:
        artifacts = train_organ_from_config(tmp_config_path, pretrained_path=pretrained_path)
        all_artifacts.append((fold, artifacts))
        print(f"\n✓ Fold {fold} training completed!")
        print(f"  Best checkpoint: {artifacts.best_ckpt}")
        print(f"  Metrics CSV: {artifacts.metrics_csv}")
    except Exception as e:
        print(f"\n✗ Fold {fold} training failed: {e}")
        import traceback
        traceback.print_exc()
        continue
    finally:
        if os.path.exists(tmp_config_path):
            os.unlink(tmp_config_path)

print(f"\n{'='*60}")
print("5-Fold Cross-Validation Training Summary (pretrained)")
print(f"{'='*60}")
print(f"Successfully trained: {len(all_artifacts)}/5 folds")
for fold, artifacts in all_artifacts:
    print(f"  Fold {fold}: {artifacts.best_ckpt}")

fold_artifacts_pretrained = {fold: artifacts for fold, artifacts in all_artifacts}

Model: rrunet3d with 3 output channels (multi-class)
Pretrained weights: models/organ/model.ts
Training all 5 folds...


Starting Fold 0 Training (pretrained)
Train CSV: annotations/splits/fold0_train.csv
Val CSV: annotations/splits/fold0_val.csv
Output directory: experiments/organ/fold0

Using device: cuda


Loading dataset: 100%|██████████| 26/26 [00:03<00:00,  6.65it/s]


Loaded pretrained weights from: models/organ/model.ts
Epoch 1/200


`torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
`torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.


  step 10/65 - loss: 1.2370
  step 20/65 - loss: 1.2322
  step 30/65 - loss: 1.2378
  step 40/65 - loss: 1.2397
  step 50/65 - loss: 1.2374
  step 60/65 - loss: 1.2370
  val mean dice: 0.7400
Epoch 2/200
  step 10/65 - loss: 1.2366
  step 20/65 - loss: 1.2377
  step 30/65 - loss: 1.2440
  step 40/65 - loss: 1.2455
  step 50/65 - loss: 1.2397
  step 60/65 - loss: 1.2355
  val mean dice: 0.7409
Epoch 3/200
  step 10/65 - loss: 1.2330
  step 20/65 - loss: 1.2391
  step 30/65 - loss: 1.2410
  step 40/65 - loss: 1.2436
  step 50/65 - loss: 1.2396
  step 60/65 - loss: 1.2373
  val mean dice: 0.7336
Epoch 4/200
  step 10/65 - loss: 1.2381
  step 20/65 - loss: 1.2354
  step 30/65 - loss: 1.2383
  step 40/65 - loss: 1.2396
  step 50/65 - loss: 1.2371
  step 60/65 - loss: 1.2330
  val mean dice: 0.7390
Epoch 5/200
  step 10/65 - loss: 1.2268
  step 20/65 - loss: 1.2338
  step 30/65 - loss: 1.2377
  step 40/65 - loss: 1.2384
  step 50/65 - loss: 1.2374
  step 60/65 - loss: 1.2358
  val mean dice:

KeyboardInterrupt: 

In [ ]:
#!/usr/bin/env python3
"""
Check which patients are in Fold 0 training and validation sets.
Run this in your notebook or terminal.
"""

import pandas as pd
from pathlib import Path

# Read the main train.csv to see fold assignments
train_csv = "data/train.csv"
df = pd.read_csv(train_csv)

# Filter to training patients only (fold >= 0)
train_df = df[df['fold'] >= 0].copy()

print("="*60)
print("FOLD 0 - Training and Validation Split")
print("="*60)
print(f"\nTotal training patients (all folds): {len(train_df)}")
print(f"Fold distribution:")
print(train_df['fold'].value_counts().sort_index().to_dict())

# Fold 0: Validation = patients with fold == 0, Training = all others
fold = 0
val_mask = train_df['fold'] == fold
val_ids = sorted(train_df[val_mask]['ID'].tolist())
train_mask = train_df['fold'] != fold
train_ids = sorted(train_df[train_mask]['ID'].tolist())

print(f"\n{'='*60}")
print(f"FOLD {fold} SPLIT:")
print(f"{'='*60}")
print(f"Training patients: {len(train_ids)}")
print(f"Validation patients: {len(val_ids)}")
print(f"Total: {len(train_ids) + len(val_ids)}")

print(f"\n{'='*60}")
print(f"VALIDATION PATIENTS (Fold {fold}):")
print(f"{'='*60}")
for i, pid in enumerate(val_ids, 1):
    print(f"{i:3d}. {pid}")

print(f"\n{'='*60}")
print(f"TRAINING PATIENTS (Fold {fold} - all other folds):")
print(f"{'='*60}")
for i, pid in enumerate(train_ids, 1):
    print(f"{i:3d}. {pid}")
    if i >= 50 and len(train_ids) > 50:
        print(f"     ... and {len(train_ids) - 50} more")
        break

# Also check the fold-specific CSV files if they exist
splits_dir = Path("annotations/splits")
fold0_train_csv = splits_dir / "fold0_train.csv"
fold0_val_csv = splits_dir / "fold0_val.csv"

if fold0_train_csv.exists() and fold0_val_csv.exists():
    print(f"\n{'='*60}")
    print("VERIFICATION: Fold-specific CSV files")
    print(f"{'='*60}")
    fold0_train_df = pd.read_csv(fold0_train_csv)
    fold0_val_df = pd.read_csv(fold0_val_csv)
    
    csv_train_ids = sorted(fold0_train_df['subject_id'].tolist())
    csv_val_ids = sorted(fold0_val_df['subject_id'].tolist())
    
    print(f"\nFrom fold0_train.csv: {len(csv_train_ids)} patients")
    print(f"From fold0_val.csv: {len(csv_val_ids)} patients")
    
    # Check if they match
    if set(csv_train_ids) == set(train_ids):
        print("✓ fold0_train.csv matches expected training set")
    else:
        print("✗ fold0_train.csv does NOT match expected training set")
        print(f"  Missing: {set(train_ids) - set(csv_train_ids)}")
        print(f"  Extra: {set(csv_train_ids) - set(train_ids)}")
    
    if set(csv_val_ids) == set(val_ids):
        print("✓ fold0_val.csv matches expected validation set")
    else:
        print("✗ fold0_val.csv does NOT match expected validation set")
        print(f"  Missing: {set(val_ids) - set(csv_val_ids)}")
        print(f"  Extra: {set(csv_val_ids) - set(val_ids)}")
else:
    print(f"\n⚠ Fold-specific CSV files not found:")
    print(f"  {fold0_train_csv}")
    print(f"  {fold0_val_csv}")
    print("  Run: python scripts/create_fold_csvs.py")

FOLD 0 - Training and Validation Split

Total training patients (all folds): 120
Fold distribution:
{0: 24, 1: 24, 2: 24, 3: 24, 4: 24}

FOLD 0 SPLIT:
Training patients: 96
Validation patients: 24
Total: 120

VALIDATION PATIENTS (Fold 0):
  1. ProstateX-0003
  2. ProstateX-0013
  3. ProstateX-0016
  4. ProstateX-0018
  5. ProstateX-0027
  6. ProstateX-0032
  7. ProstateX-0033
  8. ProstateX-0038
  9. ProstateX-0044
 10. ProstateX-0048
 11. ProstateX-0049
 12. ProstateX-0054
 13. ProstateX-0079
 14. ProstateX-0095
 15. ProstateX-0102
 16. ProstateX-0103
 17. ProstateX-0112
 18. ProstateX-0132
 19. ProstateX-0147
 20. ProstateX-0168
 21. ProstateX-0169
 22. ProstateX-0177
 23. ProstateX-0181
 24. ProstateX-0191

TRAINING PATIENTS (Fold 0 - all other folds):
  1. ProstateX-0000
  2. ProstateX-0004
  3. ProstateX-0005
  4. ProstateX-0009
  5. ProstateX-0014
  6. ProstateX-0015
  7. ProstateX-0017
  8. ProstateX-0019
  9. ProstateX-0020
 10. ProstateX-0021
 11. ProstateX-0024
 12. ProstateX

In [ ]:
# Visualize training metrics from all folds
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

base_exp_dir = Path("experiments/organ")
all_metrics = {}

# Load metrics from each fold
for fold in range(5):
    fold_dir = base_exp_dir / f"fold{fold}"
    if not fold_dir.exists():
        continue
    
    # Find the latest experiment directory
    exp_dirs = sorted(fold_dir.glob("exp-*"))
    if not exp_dirs:
        continue
    
    latest_exp = exp_dirs[-1]
    metrics_csv = latest_exp / "metrics.csv"
    
    if metrics_csv.exists():
        df = pd.read_csv(metrics_csv)
        df['fold'] = fold
        all_metrics[fold] = df
        print(f"✓ Loaded metrics for fold {fold}: {len(df)} epochs")

if not all_metrics:
    print("No metrics.csv files found. Run training first.")
else:
    # Combine all folds
    combined_df = pd.concat(all_metrics.values(), ignore_index=True)
    
    # Plot 1: Training loss per fold
    plt.figure(figsize=(15, 10))
    
    plt.subplot(2, 2, 1)
    for fold in sorted(all_metrics.keys()):
        df = all_metrics[fold]
        plt.plot(df['epoch'], df['train_loss'], label=f'Fold {fold}', alpha=0.7)
    plt.title('Training Loss per Fold')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)
    
    # Plot 2: Validation DICE per fold
    plt.subplot(2, 2, 2)
    for fold in sorted(all_metrics.keys()):
        df = all_metrics[fold]
        if 'val_dice' in df.columns:
            plt.plot(df['epoch'], df['val_dice'], label=f'Fold {fold}', alpha=0.7)
    plt.title('Validation DICE per Fold')
    plt.xlabel('Epoch')
    plt.ylabel('DICE')
    plt.legend()
    plt.grid(True)
    
    # Plot 3: Average training loss across folds
    plt.subplot(2, 2, 3)
    max_epochs = max(len(df) for df in all_metrics.values())
    epochs = np.arange(1, max_epochs + 1)
    avg_loss = []
    std_loss = []
    
    for epoch in epochs:
        epoch_losses = []
        for df in all_metrics.values():
            if epoch <= len(df):
                epoch_losses.append(df.iloc[epoch-1]['train_loss'])
        if epoch_losses:
            avg_loss.append(np.mean(epoch_losses))
            std_loss.append(np.std(epoch_losses))
        else:
            avg_loss.append(np.nan)
            std_loss.append(np.nan)
    
    plt.plot(epochs[:len(avg_loss)], avg_loss, 'b-', label='Mean', linewidth=2)
    plt.fill_between(epochs[:len(avg_loss)], 
                     np.array(avg_loss) - np.array(std_loss),
                     np.array(avg_loss) + np.array(std_loss),
                     alpha=0.3, label='±1 std')
    plt.title('Average Training Loss (across folds)')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)
    
    # Plot 4: Average validation DICE across folds
    plt.subplot(2, 2, 4)
    avg_dice = []
    std_dice = []
    
    for epoch in epochs:
        epoch_dices = []
        for df in all_metrics.values():
            if epoch <= len(df) and 'val_dice' in df.columns:
                val_dice = df.iloc[epoch-1]['val_dice']
                if pd.notna(val_dice):
                    epoch_dices.append(val_dice)
        if epoch_dices:
            avg_dice.append(np.mean(epoch_dices))
            std_dice.append(np.std(epoch_dices))
        else:
            avg_dice.append(np.nan)
            std_dice.append(np.nan)
    
    plt.plot(epochs[:len(avg_dice)], avg_dice, 'g-', label='Mean', linewidth=2)
    plt.fill_between(epochs[:len(avg_dice)],
                     np.array(avg_dice) - np.array(std_dice),
                     np.array(avg_dice) + np.array(std_dice),
                     alpha=0.3, label='±1 std')
    plt.title('Average Validation DICE (across folds)')
    plt.xlabel('Epoch')
    plt.ylabel('DICE')
    plt.legend()
    plt.grid(True)
    
    plt.tight_layout()
    plt.show()
    
    # Summary statistics
    print("\n" + "="*60)
    print("Summary Statistics (Final Epoch)")
    print("="*60)
    for fold in sorted(all_metrics.keys()):
        df = all_metrics[fold]
        final_row = df.iloc[-1]
        print(f"\nFold {fold}:")
        print(f"  Final Train Loss: {final_row['train_loss']:.4f}")
        if 'val_dice' in df.columns:
            print(f"  Final Val DICE:   {final_row['val_dice']:.4f}")
    
    # Overall statistics
    final_losses = [all_metrics[f].iloc[-1]['train_loss'] for f in sorted(all_metrics.keys())]
    final_dices = []
    for f in sorted(all_metrics.keys()):
        df = all_metrics[f]
        if 'val_dice' in df.columns:
            dice = df.iloc[-1]['val_dice']
            if pd.notna(dice):
                final_dices.append(dice)
    
    if final_dices:
        print(f"\nOverall (across {len(final_dices)} folds):")
        print(f"  Mean Val DICE: {np.mean(final_dices):.4f} ± {np.std(final_dices):.4f}")
        print(f"  Best Val DICE: {np.max(final_dices):.4f} (Fold {np.argmax(final_dices)})")


No metrics.csv files found. Run training first.
